In [1]:
import requests, csv
from bs4 import BeautifulSoup
import re
import tqdm
import pandas as pd
from collections import OrderedDict
import tqdm
from urllib.request import Request, urlopen
import os
import json
import undetected_chromedriver as uc
import time
import random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

In [3]:
DATA_DIR = Path("movie_collect/static/data/")

In [4]:
def parse_json_ld(soup):
    script_tag = soup.find('script', {'type': 'application/ld+json'})
    try:
        return json.loads(script_tag.string) if script_tag else {}
    except:
        return {}

def parse_duration(duration_str):
    if not duration_str: return 0
    duration = str(duration_str).replace("PT", "")
    hours = int(duration.split("H")[0]) if "H" in duration else 0
    minutes_part = duration.split("H")[1] if "H" in duration else duration
    minutes = int(minutes_part.split("M")[0]) if "M" in duration else 0
    return hours * 60 + minutes


def get_production_info(soup):
    companies = soup.find_all('a', {'href': re.compile('company')})
    
    if not companies:
        return [], []
        
    # Sécurisation avec un test sur le premier élément
    if "Production compan" not in companies[0].text:
        prod_names = []
    else:
        prod_names = [c.text for c in companies if "IMDbPro" not in c.text and c.text != "" and "Production compan" not in c.text]
        
    prod_codes = [str(c).split("company/")[1].split("/?ref")[0] for c in companies if "company/" in str(c)]
        
    return prod_names, prod_codes
    


def get_cast_and_director(soup):
    cast_names = []
    cast_codes = []
    cast_voice = []
    directors = []
    dir_codes = []

    persons = ["Cast","Director"]

    for person in persons:
        cast_title = soup.find(lambda tag: tag.name == "h3" and person in tag.get_text())
        if cast_title:
            cast_section = cast_title.find_parent("section")
            if cast_section:
                cast_items = cast_section.find_all("li", {"data-testid": "name-credits-list-item"})
                for item in cast_items:
                    person_link = item.find("a", class_="ipc-link ipc-link--base name-credits--title-text name-credits--title-text-big")
                    if person_link:
                        person_name = person_link.get_text(strip=True)
                        href = person_link.get("href", "")
            
                        # Code IMDb (nm...)
                        match = re.search(r"nm\d+", href)
                        person_code = match.group(0) if match else None
            
                        if person_code and person == "Cast":
                            # 2. Extraction du type (Voice / Uncredited / Aucun)
                            # On récupère tout le texte de la zone de métadonnées du personnage
                            metadata_div = item.find("div", class_="sc-ed2abe21-7 hujYBw")
                            metadata_text = (metadata_div.get_text().lower() if metadata_div else "")
        
                            if "voice" in metadata_text:
                                voice_type = "V"
                            elif "uncredited" in metadata_text:
                                voice_type = "U"
                            else:
                                voice_type = "A"
        
                            cast_names.append(person_name)
                            cast_codes.append(person_code)
                            cast_voice.append(voice_type)
                        elif person_code and person == "Director":
                            directors.append(person_name)
                            dir_codes.append(person_code)
    
    if len(cast_names) > 30:
        cast_names, cast_codes, cast_voice = cast_names[:30], cast_codes[:30], cast_voice[:30]

    return cast_names, cast_codes, cast_voice, directors, dir_codes



# --- FONCTION PRINCIPALE ---

def movie_imdb_full_logic(imdb_id, french_title, driver):
    
    # 1. PAGE PRINCIPALE (Infos, Budget, Directors)
    url_main = f"https://www.imdb.com/title/{imdb_id}/?language=en-US"
    driver.get(url_main)
    time.sleep(random.uniform(4, 6))
        
    soup_main = BeautifulSoup(driver.page_source, "html.parser")
    info_main = parse_json_ld(soup_main)
        
    # 2. PAGE FULL CREDITS (Pour avoir le Top 30)
    url_credits = f"https://www.imdb.com/title/{imdb_id}/fullcredits?language=en-US"
    driver.get(url_credits)
    time.sleep(random.uniform(3, 5))
        
    soup_credits = BeautifulSoup(driver.page_source, "html.parser")

    # --- EXTRACTION ---
    title = info_main.get("name", "").replace("&amp;", "&").replace("&apos;", "'")

    # Cast (depuis la fonction dédiée sur soup_credits)
    cast_names, cast_codes, cast_roles, directors, dir_codes = get_cast_and_director(soup_credits)

    # Le reste de la logique (Budget, Pays, Langues, Prods)
    # Budget

    budget, bud_currency = 0, ""
    
    try :
        budget_label = soup_main.find(lambda tag: tag.name == 'span' and tag.text.strip() == 'Budget')
        if budget_label:
            # 2. Récupérer le span suivant qui contient la valeur ($63,000,000...)
            budget_value_span = budget_label.find_next(
                'span', 
                {'class': re.compile('ipc-metadata-list-item__list-content-item')}
            )
            if budget_value_span:
                budget_text = budget_value_span.text.strip()
                
            parts = budget_text.replace("\u202f", "").replace("\xa0", " ").split(" ")
    
            try:
                budget = int(parts[0][1:].replace(",",""))
                bud_currency = parts[0][0]
            except ValueError:
                budget = int(parts[1].replace(",",""))
                bud_currency = parts[0]
    except Exception:
        budget, bud_currency = 0, ""

    # Productions
    prod_names, prod_codes = get_production_info(soup_main)

    # Rating, Year, Genres, Pays, Langues
    rating = info_main.get("aggregateRating", {}).get("ratingValue", "")
    num_rate = info_main.get("aggregateRating", {}).get("ratingCount", "")

    year_m = re.search(r'\((\d{4})\)', soup_main.title.text)
    year = int(year_m.group(1)) if year_m else None

    duration = parse_duration(info_main.get("duration", ""))
    genres = info_main.get("genre", [])

    country = [c.text for c in soup_main.find_all('a', {'href': re.compile('country_of_origin')})] or ["No Info"]
    country = ["USA" if c == "United States" else "UK" if c == "United Kingdom" else c for c in country]
    language = [l.text for l in soup_main.find_all('a', {'href': re.compile('primary_language')})] or ["No Info"]

    return (
            imdb_id, french_title, title, year, directors, dir_codes, 
            cast_names, cast_codes, cast_roles, genres, duration, 
            country, language, prod_names, prod_codes, 
            rating, num_rate, budget, bud_currency
        )


In [5]:
# import movies to scrap
df = pd.read_excel(DATA_DIR / 'scrap_it.xlsx')

In [6]:
df["query"] = df.apply(lambda x : (x["imdb"],x["title"]), axis=1)

In [7]:
db_df = pd.read_csv(DATA_DIR / 'db_backup/movies_db.csv', encoding="utf-8-sig", sep=";")

In [8]:
imdb_db = db_df["movie_id"].values
final_imdb = [(i,j) for (i,j) in list(df["query"].values) if i not in imdb_db]

In [9]:
final_imdb = [("tt28443655","Death of a Unicorn")]

In [10]:
movies = []

In [11]:
driver = webdriver.Chrome()
chrome_version = driver.capabilities["browserVersion"]
print("Version de Chrome (via Selenium) :", chrome_version)

driver.quit()

Version de Chrome (via Selenium) : 157.0.8103.79


In [12]:
options = uc.ChromeOptions()
    # options.add_argument('--headless') 

driver = uc.Chrome(options=options, version_main=int(chrome_version.split(".")[0]), use_subprocess=True)
driver.execute_cdp_cmd("Network.enable", {})

driver.execute_cdp_cmd("Network.setExtraHTTPHeaders", {
        "headers": {
            "Accept-Language": "en-US,en;q=0.9"
        }
    })

{}

In [13]:
try :
    for (i, j) in tqdm.tqdm(final_imdb):
        movies.append(movie_imdb_full_logic(i, j, driver))
finally :
    driver.quit()

100%|████████████████████████████████████████████| 1/1 [00:12<00:00, 12.46s/it]


In [14]:
movie_df = pd.DataFrame(movies,columns = ["movie_id","french_title","original_title","year","director","director_id",\
                                          "actor","actor_id","status","genre","duration","country",\
                                          "language","production","prod_id","rating","num_rate",\
                                          "budget","currency"])

In [15]:
# keep movies with a rating (the other are probably not available)
movie_keep = movie_df.drop(movie_df[movie_df["rating"] == ""].index).reset_index(drop=True)
too_soon = pd.DataFrame(movie_df[["movie_id","french_title"]].drop(movie_df[movie_df["rating"] != ""].index).values, \
            columns= ['imdb_id', 'title']).set_index("imdb_id")

In [16]:
# save scraped movies with no ratings
if len(too_soon) > 0:
    f_scrap = pd.read_csv(DATA_DIR / 'too_soon.csv', encoding="utf-8-sig", sep=";").set_index("imdb_id")
    too_soon = pd.concat([f_scrap, too_soon])
    too_soon.to_csv(DATA_DIR / 'too_soon.csv', encoding="utf-8-sig", sep=";")

In [17]:
changer = {"$US":1, "$":1, "$CA":0.72, "$AU":0.65, "€":1.08, "£GB":1.29, "₩":0.00073, "₹":0.012, "CNY":0.14, "RUR":0.011, \
          "CZK":0.043, "NOK":0.092, "BDT":0.0085, "HKD":0.13, "R$":0.162, "CHF":0.8, "MYR":0.24}

In [18]:
movie_keep["def_budget"] = movie_keep["budget"] * movie_keep["currency"].apply(lambda x : changer[x] if x != "" else 1)

In [19]:
movie_keep["def_budget"] = movie_keep["def_budget"].apply(lambda x: int(x) if x !=0 else "")

In [20]:
modifs = ["actor","country","director","genre","language","production"]

In [21]:
for mod in modifs:
    movie_keep[f"{mod}s"] = movie_keep[mod].apply(lambda x : " | ".join(x))

In [22]:
movie_keep["saw"] = False
movie_keep["wishlist"] = False

In [23]:
select = movie_keep[["movie_id","saw","wishlist","french_title","original_title","year","directors","actors","genres",\
                     "duration","countrys","languages","productions","rating","num_rate","def_budget"]]

In [24]:
select.to_excel(DATA_DIR / "movies_scraped.xlsx", index=False)

In [25]:
# update movies_db
up_movies = select[["movie_id","french_title","original_title","year","duration","rating","num_rate","def_budget"]]\
                    .set_index("movie_id").rename(columns = {"def_budget":"budget"})
up_movies.to_csv(DATA_DIR / "update_db/up_movies.csv", encoding="utf-8-sig", sep=";")

In [26]:
# update user_db
up_user = select[["movie_id","saw","wishlist"]].set_index("movie_id")
up_user.to_csv(DATA_DIR / "update_db/up_user.csv", encoding="utf-8-sig", sep=";")

In [27]:
genre_df = pd.read_csv(DATA_DIR / "db_backup/genres_db.csv", sep=";")

In [28]:
genre_dict = dict(zip(genre_df["name"], genre_df["genre_id"]))
genre_dict[""] = ""

In [29]:
tot_genre_df = movie_keep[["movie_id","genre"]].explode(["genre"], ignore_index=True)
tot_genre_df = tot_genre_df.fillna("")
tot_genre_df["genre_id"] = tot_genre_df["genre"].apply(lambda x: genre_dict[x])
tot_genre_df = tot_genre_df.drop(["genre"], axis=1).drop_duplicates()
tot_genre_df = tot_genre_df[tot_genre_df["genre_id"].notna() & (tot_genre_df["genre_id"].astype(str).str.strip() != "")]

In [30]:
tot_genre_df.set_index("movie_id").to_csv(DATA_DIR / "update_db/up_movie_genre.csv", sep=";")

### Update actors

In [51]:
tot_actor_df = movie_keep[["movie_id","actor", "actor_id", "status"]].explode(["actor", "actor_id", "status"], ignore_index=True)
tot_actor_df["status"] = tot_actor_df["status"].str.replace("A", "", regex=False)
tot_actor_final = tot_actor_df.drop(["actor"], axis=1).drop_duplicates()
tot_actor_final = tot_actor_final[tot_actor_final["actor_id"].notna() & (tot_actor_final["actor_id"].str.strip() != "")]
tot_actor_final.set_index("movie_id").to_csv(DATA_DIR / "update_db/up_movie_actor.csv", sep=";")

In [52]:
act_db = pd.read_csv(DATA_DIR / f'db_backup/actors_db.csv', encoding="utf-8-sig", sep=";")

In [53]:
actors_df = tot_actor_df.drop(["movie_id", "status"], axis=1).drop_duplicates()

In [ ]:
def update_actors(movie_keep):
    tot_actor_df = movie_keep[["movie_id","actor", "actor_id", "status"]].explode(["actor", "actor_id", "status"], ignore_index=True)
    tot_actor_df["status"] = tot_actor_df["status"].str.replace("A", "", regex=False)
    tot_actor_final = tot_actor_df.drop(["actor"], axis=1).drop_duplicates()
    tot_actor_final = tot_actor_final[tot_actor_final["actor_id"].notna() & (tot_actor_final["actor_id"].str.strip() != "")]
    tot_actor_final.set_index("movie_id").to_csv(DATA_DIR / "update_db/up_movie_actor.csv", sep=";")

    act_db = pd.read_csv(DATA_DIR / f'db_backup/actors_db.csv', encoding="utf-8-sig", sep=";")

    actors_df = tot_actor_df.drop(["movie_id", "status"], axis=1).drop_duplicates()
    new_actors = actors_df[~actors_df["actor_id"].isin(act_db["actor_id"])]
    new_actors= new_actors[new_actors["actor_id"].notna() & (new_actors["actor_id"].str.strip() != "")]
    if len(new_actors) != 0:
        new_actors.columns = ["name","actor_id"]
        new_actors.set_index("actor_id").to_csv(DATA_DIR / "update_db/up_actor.csv", encoding="utf-8-sig", sep=";")

In [54]:
act_db = pd.read_csv(DATA_DIR / f'db_backup/actors_db.csv', encoding="utf-8-sig", sep=";")

actors_df = tot_actor_df.drop(["movie_id", "status"], axis=1).drop_duplicates()
new_actors = actors_df[~actors_df["actor_id"].isin(act_db["actor_id"])]
new_actors = new_actors[new_actors["actor_id"].notna() & (new_actors["actor_id"].str.strip() != "")]
if len(new_actors) != 0:
    new_actors.columns = ["name","actor_id"]
    new_actors.set_index("actor_id").to_csv(DATA_DIR / "update_db/up_actor.csv", encoding="utf-8-sig", sep=";")

### Update director and production

In [60]:
db_dict = {"dir":["director","director_id"],"prod":["production","prod_id"]}

In [66]:
for k, v in db_dict.items():
    tot_df = movie_keep[["movie_id",v[0], v[1]]].explode([v[0], v[1]], ignore_index=True)
    tot_df_final = tot_df.drop([v[0]], axis=1).drop_duplicates()
    tot_df_final = tot_df_final[tot_df_final[v[1]].notna() & (tot_df_final[v[1]].str.strip() != "")]
    tot_df_final.set_index("movie_id").to_csv(DATA_DIR / f"update_db/up_movie_{k}.csv", sep=";")

    val_db = pd.read_csv(DATA_DIR / f'db_backup/{v[0]}_db.csv', encoding="utf-8-sig", sep=";")
    
    val_df = tot_df.drop(["movie_id"], axis=1).drop_duplicates()
    new_val = val_df[~val_df[v[1]].isin(val_db[v[1]])]
    new_val = new_val[new_val[v[1]].notna() & (new_val[v[1]] != "")]
    if len(new_val) != 0:
        new_val.columns = ["name",v[1]]
        new_val.set_index(v[1]).to_csv(DATA_DIR / f"update_db/up_{v[0]}.csv", encoding="utf-8-sig", sep=";")

### Update Country and Language

In [41]:
db_list = ["country", "language"]

In [42]:
def get_or_create_id(name, current_dict, max_val):
    if name not in current_dict:
        max_val[0] += 1
        current_dict[name] = max_val[0]
    return current_dict[name]

In [43]:
for val in db_list:
    try:
        val_db2 = pd.read_csv(DATA_DIR / f'db_backup/{val}_db.csv', encoding="ISO-8859-1", sep=";")
        info_dict = dict(zip(val_db2["name"], val_db2[f"{val}_id"]))
    except KeyError:
        val_db2 = pd.read_csv(DATA_DIR / f'db_backup/{val}_db.csv', encoding="utf-8-sig", sep=";")
        info_dict = dict(zip(val_db2["name"], val_db2[f"{val}_id"]))
    info_dict = { (None if pd.isna(k) else k): v for k, v in info_dict.items() }
    current_max_id = val_db2[f"{val}_id"].max()

    max_ref = [current_max_id]
    
    tot_df2 = movie_keep[["movie_id", val]].explode([val], ignore_index=True)
    tot_df2[f"{val}_id"] = tot_df2[val].apply(lambda x: get_or_create_id(x, info_dict, max_ref))
    tot_df2.columns = ["movie_id","name",f"{val}_id"]
    tot_df2_final = tot_df2.drop("name", axis=1).drop_duplicates()
    tot_df2_final = tot_df2_final[tot_df2_final[f"{val}_id"].notna() & (tot_df2_final[f"{val}_id"] != "")]
    tot_df2_final.set_index("movie_id").to_csv(DATA_DIR / f"update_db/up_movie_{val}.csv", encoding="utf-8-sig", sep=";")
    
    new_val2 = tot_df2[~tot_df2[f"{val}_id"].isin(val_db2[f"{val}_id"])]
    new_val2 = new_val2[new_val2[f"{val}_id"].notna() & (new_val2[f"{val}_id"] != "")]
    new_val2 = new_val2.drop(["movie_id"], axis=1).drop_duplicates()
    if len(new_val2) != 0:
        new_val2.set_index(f"{val}_id").to_csv(DATA_DIR / f"update_db/up_{val}.csv", encoding="utf-8-sig", sep=";")

In [44]:
import sqlalchemy as db
import psycopg2
import pandas as pd
import os

In [45]:
from dotenv import load_dotenv

In [46]:
load_dotenv()

True

In [47]:
DATABASE_URL = os.getenv("DATABASE_URL")

In [48]:
engine = db.create_engine(DATABASE_URL)
connection = engine.connect()

In [70]:
dict_file = {"movies_db":["up_movies", "movie_id", "movies"], "actors_db":["up_actor", "actor_id", "actors"], \
             "country_db":["up_country", "country_id", "country"], "director_db":["up_director", "director_id","directors"], \
             "genres_db":["up_genre", "genre_id", "genres"], "language_db":["up_language", "language_id", "language"], \
             "production_db":["up_production", "prod_id", "production"],"movie_actor_db_2":["up_movie_actor", "movie_id", "movie_actor"], \
             "movie_country_db":["up_movie_country", "movie_id", "movie_country"], "movie_dir_db":["up_movie_dir", "movie_id", "movie_director"], \
             "movie_genre_db":["up_movie_genre", "movie_id", "movie_genre"], "movie_language_db":["up_movie_language", "movie_id", "movie_language"], \
             "movie_prod_db":["up_movie_prod", "movie_id", "movie_prod"], "user_list_db":["up_user", "movie_id", "user_list"]}

In [69]:
for key, value in dict_file.items():
    if os.path.isfile(DATA_DIR / f"update_db/{value[0]}.csv"):
        #database = pd.read_csv(DATA_DIR / f"db_backup/{key}.csv",encoding="utf-8-sig", sep=";").set_index(value[1])
        up = pd.read_csv(DATA_DIR / f"update_db/{value[0]}.csv",encoding="utf-8-sig", sep=";").set_index(value[1])
        for col in up.select_dtypes(include=["object"]).columns:
            up[col] = up[col].astype(str).str.replace("’", "'", regex=False)
        up.to_sql(value[2], engine, if_exists="append")
        #pd.concat([database, up]).to_csv(DATA_DIR / f"db_backup/{key}.csv",encoding="utf-8-sig", sep=";")